# Tool

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()


os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [4]:

model = init_chat_model("groq:qwen/qwen3-32b")
res = model.invoke("why do parrots talk?")
res

AIMessage(content='<think>\nOkay, so the user is asking why parrots talk. Hmm, I need to break this down. First, I know parrots are known for mimicking human speech, but why exactly do they do that? Let me think. Maybe it\'s related to their social behavior? I remember reading somewhere that parrots are social animals, so perhaps talking is a way to interact with their environment or other parrots. But wait, when they talk, they\'re imitating humans, not other parrots. So maybe they learn from their human owners?\n\nAlso, there\'s the aspect of vocal learning. Not all birds can talk, but parrots can. What\'s different about their anatomy? I think their syrinx, the vocal organ, is more complex. But how does that help them mimic speech? Maybe their brain structure is specialized for this. I remember something about a part of the brain called the song control system, which is involved in learning and producing vocalizations.\n\nAnother angle: why would parrots evolve this ability? If they

In [ ]:
from langchain.tools import tool

@tool 
def get_weather(location:str)-> str:
    """Get the weather at a location"""
    return f"It's sunny in { location}"

model_with_tool = model.bind_tools([get_weather])

In [11]:
res = model_with_tool.invoke("what is a weather like in boston")
print(res)

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. Let me check if I have the right tool for that. The available function is get_weather, which requires a location parameter. Boston is a valid location, so I should call that function. I need to make sure to format the response correctly with the tool_call tags. Let me structure the JSON with the name as get_weather and the arguments as location: Boston. Double-checking the syntax to avoid errors.\n', 'tool_calls': [{'id': '1xmktkhb6', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 118, 'prompt_tokens': 154, 'total_tokens': 272, 'completion_time': 0.203261966, 'completion_tokens_details': {'reasoning_tokens': 94}, 'prompt_time': 0.006275556, 'prompt_tokens_details': None, 'queue_time': 0.057541493, 'total_time': 0.209537522}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint':

In [13]:
for tool_call in res.tool_calls:
    print(f"Tools : {tool_call['name']}")
    print(f"Args : {tool_call['args']}")

Tools : get_weather
Args : {'location': 'Boston'}


# Tool Execution Loops


In [ ]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tool.invoke(messages)

messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
  # Execute the tool with the generated arguments
  tool_result = get_weather.invoke(tool_call)
  messages.append(tool_result)
  
# Step 3: Pass results back to model for final response
final_response = model_with_tool.invoke(messages)

print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Boston, I need to call this function with "Boston" as the location. I\'ll make sure the parameters are correctly formatted in JSON. No other tools are available, so this should be the only function call needed.\n', 'tool_calls': [{'id': '3zt028t06', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 104, 'prompt_tokens': 153, 'total_tokens': 257, 'completion_time': 0.170503737, 'completion_tokens_details': {'reasoning_tokens': 80}, 'prompt_time': 0.00706116, 'prompt_tokens_details': None, 'queue_time': 0.051080459, 'total_time': 0.177564897}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason

In [19]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Boston, I need to call this function with "Boston" as the location. I\'ll make sure the parameters are correctly formatted in JSON. No other tools are available, so this should be the only function call needed.\n', 'tool_calls': [{'id': '3zt028t06', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 104, 'prompt_tokens': 153, 'total_tokens': 257, 'completion_time': 0.170503737, 'completion_tokens_details': {'reasoning_tokens': 80}, 'prompt_time': 0.00706116, 'prompt_tokens_details': None, 'queue_time': 0.051080459, 'total_time': 0.177564897}, 'model_name': 'qwen/qwen3-32b', 'syste